In [5]:
# === Librerías == #
# Importamos las clases que se requieren para manejar los agentes (Agent)
# y su entorno (Model). Cada modelo puede contener múltiples agentes.
from mesa import Agent, Model
from mesa.space import MultiGrid

# Para este problema, usaremos un espacio continuo.
from mesa.space import ContinuousSpace

# Haremos uso de ''DataCollector'' para obtener información de cada paso
# de la simulación.
from mesa.datacollection import DataCollector

# matplotlib lo usaremos crear una animación de cada uno de los pasos
# del modelo.
%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.animation as animation
plt.rcParams["animation.html"] = "jshtml"
matplotlib.rcParams['animation.embed_limit'] = 2**128

# Importamos los siguientes paquetes para el mejor manejo de valores
# numéricos.
import numpy as np
import pandas as pd

In [6]:
# ==== Modelo === #

# ==== Clases de modelo ==== #

class Nodo():
    def __init__(self,pos,enter_value):
        self.pos = pos # Vector2D de posicion (x,y)
        self.enter_value = enter_value #Valor AP de entrada a la celda
        self.contenido = None #Objeto complejo de POE 
        self.vecinos = {} #Inicializar diccionario de vecinos

    def obtener_costo(self):
        return self.enter_value

In [7]:
from enum import Enum

# Usar enums para que sea bonito y no estar usando strings feos.
class TipoArista(Enum):
    MURO = 1
    PUERTA = 2


class Arista():
    def __init__(self,tipo):
        self.tipo = tipo #Enum de tipo de Arista (puerta, pared)
        self.value = 0

    def obtener_costo(self):
        return self.value

class Puerta(Arista):
    def __init__(self):
        super().__init__(TipoArista.PUERTA)
        self.cerrado = True

    def abrir(self):
        self.cerrado = False

    def obtener_costo(self):
        return int(self.cerrado) # si esta cerrado devolvera valor de 1, si esta abierto devolvera valor de 0


class Muro(Arista):
    def __init__(self):
        super().__init__(TipoArista.MURO)
        self.hp = 2

    def golpear(self):
        if(self.hp > 0):
            self.hp -= 1

    def obtener_costo(self):
        valor = (self.hp * 2)
        return valor

    

In [29]:
class FlashPointModel(Model):
    def __init__(self, numAgents, width, height):
        super().__init__()
        self.grid = MultiGrid(width, height, torus=False) # Grid de vectores2D

        ## Data collector (despues lo hacemos) ##

        ## === Crear matriz de Nodos y llenar la grid de Mesa ==

        # 1. Crear los Nodos de cada casilla segun dimensiones
        self.mapa_nodos = {} # mapa_nodos será nuestro contenedor global que funciona como indice general. Mete una coordenada y recibes un Nodo
        for x in range(width):
            for y in range(height):
                nodo = Nodo(pos=(x, y), enter_value=1)
                self.mapa_nodos[(x, y)] = nodo

        # 2. Inicializamos relaciones ortogonales con objetos vacíos
        self._conectar_vecinos_base(width, height)

        # 3. Colocar Muros y Puertas específicas del mapa de Flash Point
        self._cargar_infraestructura_tablero()

    def _conectar_vecinos_base(self, width, height):
        for (x, y), nodo in self.mapa_nodos.items():
            # Direcciones ortogonales
            direcciones = [(x+1, y), (x-1, y), (x, y+1), (x, y-1)]
            # Usar nx & ny como neighbor x,y.
            for nx, ny in direcciones:
                if 0 <= nx < width and 0 <= ny < height:
                    nodo_vecino = self.mapa_nodos[(nx, ny)]
                    # Inicializa las llaves del diccionario con los Nodos vecinos
                    nodo.vecinos[nodo_vecino] = None  # Por defecto, la conexión no tiene objeto Arista (None / Paso Libre)

    def _colocar_borde(self, pos_a, pos_b, objeto_arista):
        ## Sincroniza la misma referencia de Arista Object a los Nodos adyacentes
        nodo_a = self.mapa_nodos[pos_a]
        nodo_b = self.mapa_nodos[pos_b]

        # Llenado de diccionario vecinos {Nodo:Arista}
        # Utiliza el nodo como llave en el diccionario y asigna como valor el objeto Arista
        nodo_a.vecinos[nodo_b] = objeto_arista
        nodo_b.vecinos[nodo_a] = objeto_arista

    def _cargar_infraestructura_tablero(self):
        # Definimos los planos de Objetos Aristas
        # Son arrays de vectores de vectores2D
        # Ejemplo: Pared entre ((0,0), (1,0))

        # Mapa 10x8
        lista_muros = [

            ((0,1), (1,1)),
            ((0,2), (1,2)),
            ((0,3), (1,3)),
            ((0,4), (1,4)),
            ((0,5), (1,5)),
            ((0,6), (1,6)),

            ((9,1), (8,1)),
            ((9,2), (8,2)),
            ((9,3), (8,3)),
            ((9,4), (8,4)),
            ((9,5), (8,5)),
            ((9,6), (8,6)),

            ((1,0), (1,0)),
            ((2,0), (2,0)),
            ((3,0), (3,0)),
            ((4,0), (4,0)),
            ((5,0), (5,0)),
            ((6,0), (6,0)),
            ((7,0), (7,0)),
            ((8,0), (8,0)),

            ((1,6), (1,7)),
            ((2,6), (2,7)),
            ((3,6), (3,7)),
            ((4,6), (4,7)),
            ((5,6), (5,7)),
            ((6,6), (6,7)),
            ((7,6), (7,7)),
            ((8,6), (8,7)),

            ((1,4), (1,5)),
            ((2,4), (2,5)),
            ((3,4), (3,5)),
            ((4,4), (4,5)),
            ((5,4), (5,5)),
            ((6,4), (6,5)),
            ((7,4), (7,5)),
            ((8,4), (8,5)),

            ((3,2), (3,3)),
            ((4,2), (4,3)),
            ((5,2), (5,3)),
            ((6,2), (6,3)),
            ((7,2), (7,3)),
            ((8,2), (8,3)),

            ((1,3), (1,4)),
            ((2,3), (2,4)),

            ((1,5), (1,6)),
            ((2,5), (2,6)),

            ((3,2), (3,3)),
            ((4,2), (4,3)),

            ((3,6), (3,7)),
            ((4,6), (4,7)),

            ((1,3), (1,4)),
            ((2,3), (2,4)),

            ((1,3), (1,4)),
            ((2,3), (2,4)),

            ((5,7), (5,8)),
            ((6,7), (6,8)),


        ]
        
        lista_puertas = [
            
        ]

        for pos_a, pos_b in lista_muros:
            self._colocar_borde(pos_a, pos_b, Muro())

        for pos_a, pos_b in lista_puertas:
            self._colocar_borde(pos_a, pos_b, Puerta())

    ## === Visualizacion DEBUG  === ##
    ## == Borrar al final == ##
    def imprimir_tablero_debug(self):
        """
        Dibuja en la consola la cuadrícula mostrando:
        - Nodos con su estado de fuego (0: Limpio, 1: Humo, 2: Fuego)
        - Muros intactos (║ o ═) / Muros dañados 1 HP (│ o ─)
        - Puertas cerradas (D) / Puertas abiertas (d)
        """
        print("\n" + "=" * 45)
        print("          DEBUG: MAPA DE FLASH POINT")
        print(" Leyenda: [0]=Limpio [1]=Humo [2]=Fuego")
        print("          ║/═ Muro intacto (2HP) | │/─ Muro dañado (1HP)")
        print("          D Puerta cerrada       | d Puerta abierta")
        print("=" * 45 + "\n")

        # Dibujar la cuadrícula de arriba hacia abajo
        for y in range(self.grid.height - 1, -1, -1):
            
            # 1. Renglón superior del nodo (Muros/Puertas Horizontales)
            linea_horizontal = "  "
            for x in range(self.grid.width):
                nodo_actual = self.mapa_nodos[(x, y)]
                nodo_arriba = self.mapa_nodos.get((x, y + 1))
                
                # Borde superior
                if nodo_arriba and nodo_arriba in nodo_actual.vecinos:
                    borde = nodo_actual.vecinos[nodo_arriba]
                    linea_horizontal += self._simbolo_borde_horizontal(borde) + " "
                else:
                    linea_horizontal += "─── "  # Limite exterior o paso libre
            print(linea_horizontal)

            # 2. Renglón del nodo (Nodos y Muros/Puertas Verticales)
            linea_nodos = f"{y} "
            for x in range(self.grid.width):
                nodo_actual = self.mapa_nodos[(x, y)]
                nodo_derecha = self.mapa_nodos.get((x + 1, y))

                # Renderizar estado del nodo (enter_value / estado de fuego)
                linea_nodos += f"[{nodo_actual.enter_value}]"

                # Borde derecho
                if nodo_derecha and nodo_derecha in nodo_actual.vecinos:
                    borde = nodo_actual.vecinos[nodo_derecha]
                    linea_nodos += self._simbolo_borde_vertical(borde)
                else:
                    linea_nodos += " "  # Paso libre
            print(linea_nodos)

        # Renglón inferior final y ejes X
        print("  " + "─── " * self.grid.width)
        print("   " + "   ".join(str(x) for x in range(self.grid.width)) + "\n")

    def _simbolo_borde_horizontal(self, borde):
        if borde is None:
            return "   "  # Paso libre
        if isinstance(borde, Muro):
            if borde.hp == 2:
                return "═══"
            elif borde.hp == 1:
                return "───"
            return "   "  # Destruido
        if isinstance(borde, Puerta):
            return " D " if borde.cerrado else " d "
        return "   "

    def _simbolo_borde_vertical(self, borde):
        if borde is None:
            return " "  # Paso libre
        if isinstance(borde, Muro):
            if borde.hp == 2:
                return "║"
            elif borde.hp == 1:
                return "│"
            return " "  # Destruido
        if isinstance(borde, Puerta):
            return "D" if borde.cerrado else "d"
        return " "
        
        

In [28]:
modelo = FlashPointModel(numAgents=0, width=10, height=9)
modelo.imprimir_tablero_debug()


          DEBUG: MAPA DE FLASH POINT
 Leyenda: [0]=Limpio [1]=Humo [2]=Fuego
          ║/═ Muro intacto (2HP) | │/─ Muro dañado (1HP)
          D Puerta cerrada       | d Puerta abierta

  ─── ─── ─── ─── ─── ─── ─── ─── ─── ─── 
8 [1] [1] [1] [1] [1] [1] [1] [1] [1] [1] 
                      ═══ ═══             
7 [1] [1] [1] [1] [1] [1] [1] [1] [1] [1] 
      ═══ ═══ ═══ ═══ ═══ ═══ ═══ ═══     
6 [1]║[1] [1] [1] [1] [1] [1] [1] [1]║[1] 
      ═══ ═══                             
5 [1]║[1] [1] [1] [1] [1] [1] [1] [1]║[1] 
      ═══ ═══ ═══ ═══ ═══ ═══ ═══ ═══     
4 [1]║[1] [1] [1] [1] [1] [1] [1] [1]║[1] 
      ═══ ═══                             
3 [1]║[1] [1] [1] [1] [1] [1] [1] [1]║[1] 
              ═══ ═══ ═══ ═══ ═══ ═══     
2 [1]║[1] [1] [1] [1] [1] [1] [1] [1]║[1] 
                                          
1 [1]║[1] [1] [1] [1] [1] [1] [1] [1]║[1] 
                                          
0 [1] [1] [1] [1] [1] [1] [1] [1] [1] [1] 
  ─── ─── ─── ─── ─── ─── ─── ─── ─── 